# ✅ Si funciona

In [1]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting

# Diagnóstico del entorno
!nvidia-smi | head -n 15
!nvcc --version
!python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"

# Parches de cabeceras (idempotentes)
%cd /content/gaussian-splatting
!grep -q "<cstdint>" submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h || \
  sed -i '/#pragma once/a #include <cstdint>' submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
!grep -q "<float.h>" submodules/simple-knn/simple_knn.cu || \
  sed -i '1i #include <float.h>' submodules/simple-knn/simple_knn.cu

# Compilación sin aislamiento y sin -q
import torch
cc = torch.cuda.get_device_capability()
%env TORCH_CUDA_ARCH_LIST={cc[0]}.{cc[1]}

!pip install plyfile
!pip install --no-build-isolation submodules/diff-gaussian-rasterization
!pip install --no-build-isolation submodules/simple-knn

# Verificación antes de entrenar
!python -c "import diff_gaussian_rasterization, simple_knn; print('OK')"

# Dataset
!wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!unzip -q tandt_db.zip

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 603, done.
remote: Total 603 (delta 0), reused 0 (delta 0), pack-reused 603 (from 1)
Receiving objects: 100% (603/603), 2.09 MiB | 4.90 MiB/s, done.
Resolving deltas: 100% (346/346), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (174/174), done.        
remote: Total 3293 (delta 171), reused 280 (delta 148), pack-reused 2971 (from 1)        
Receiving objects: 

In [ ]:
!python train.py -s /content/gaussian-splatting/tandt/train

2026-10-04 22:23:57.983569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing 
Output folder: ./output/f0226fdd-8 [04/10 22:24:02]
Reading camera 301/301 [04/10 22:24:04]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [04/10 22:24:04]
Loading Training Cameras [04/10 22:24:06]
Loading Test Cameras [04/10 22:24:11]
Number of points at initialisation :  182686 [04/10 22:24:11]
Training progress:   6% 1680/30000 [01:25<23:45, 19.86it/s, Loss=0.1632033]